In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'data' / '3_train.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'data' / '3_val.parquet')

# from sklearn import set_config
# set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

In [2]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

# 종속변수군
target_col = 'target_maxrise_6h'

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]

# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = [
    col for col in (train.select_dtypes(include='number').columns)
    if col not in ['event_id', target_col]]  # 'station_id', 'timestamp'는 미포함
origin_cols = wl_cols + rain_cols + env_cols

assert len(num_cols) == 31, "'num_cols' 개수 이상"  # 종속변수 제외됨
assert (set(train.columns) - set(num_cols)) == {'station_id', 'timestamp', 'event_id', target_col}, "'num_cols' 확인필요"
assert set(origin_cols).issubset(num_cols), "'origin_cols' 확인필요"

In [12]:
# ============================================================
# 2. 종속변수 분리
# ============================================================

y_train = train[target_col]
y_val = val[target_col]

In [ ]:
# ============================================================
# 3. 최종 독립변수 정의(OHE 포함)
# ============================================================

# 기존 독립변수 정의
cols = origin_cols + wl_diff_cols

# 독립변수에 OHE 추가하기
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    handle_unknown='ignore',  # train에 없던 범주가 나와도 에러 안띄움
    sparse_output=True  # 희소행렬로 반환(원핫인코딩을 가볍게 만들어줌, 빅데이터에서 필수)
)

ohe.fit(train[['station_id']])  # skleran 인코더 2차원 행렬 필수

assert set(val['station_id']).issubset(set(train['station_id'])), "val에 변환되지 않은 'station_id' 발생 가능"

train_ohe = ohe.transform(train[['station_id']])
val_ohe = ohe.transform(val[['station_id']])
# 여기까지는 변환, 실제 데이터에 합친건 아님

# 합치기
from scipy.sparse import hstack

X_train = hstack([
    train[cols],
    train_ohe
]).tocsr()

X_val = hstack([
    val[cols],
    val_ohe
]).tocsr()

# 행 개수 검증
assert X_train.shape[0] == len(train), "Train 행 개수 불일치"
assert X_val.shape[0] == len(val), "Val 행 개수 불일치"
# 열 개수 검증
assert X_train.shape[1] == len(cols) + train_ohe.shape[1], "Train 열 개수 불일치"
assert X_val.shape[1] == X_train.shape[1], "Train과 Val 열 개수 불일치"

In [16]:
# ============================================================
# 4. 모델비교
# ============================================================

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import root_mean_squared_error

# 모델 구동시간 측정용
import time

# 모델 정의
models = {
    'LightGBM': LGBMRegressor(
        verbosity=-1,
        random_state=42
    ),

    'XGBoost': XGBRegressor(
        verbosity=1,
        random_state=42
    ),

    'CatBoost': CatBoostRegressor(
        verbose=500,
        random_state=42
    ),

    'RandomForest_light': RandomForestRegressor(
        n_estimators=30,
        max_depth=20,
        max_features=0.3,
        min_samples_leaf=10,
        n_jobs=2,
        verbose=1,
        random_state=42
    ),

    'ExtraTrees_light': ExtraTreesRegressor(
        n_estimators=30,
        max_depth=20,
        max_features=0.3,
        min_samples_leaf=10,
        n_jobs=2,
        verbose=1,
        random_state=42
    )
}

# 학습, 예측, 평가
results = []

for name, model in models.items():
    
    # 학습
    start = time.time()

    model.fit(X_train, y_train)

    end = time.time()

    elapsed = end - start

    # 예측
    pred_val = model.predict(X_val)
    pred_train = model.predict(X_train)

    # RMSE 평가
    rmse_val = root_mean_squared_error(
        y_val,
        pred_val
    )
    rmse_train = root_mean_squared_error(
        y_train,
        pred_train
    )

    # 과적합 진단
    gap = rmse_val - rmse_train

    results.append({
        'Model': name,
        'Time' : elapsed,
        'val_RMSE': rmse_val,
        'train_RMSE': rmse_train,
        'Gap': gap
    })

    print(f"{name} 모델링 완료")

result_df = pd.DataFrame(results)

display(result_df)

LightGBM 모델링 완료
XGBoost 모델링 완료
Learning rate set to 0.112971
0:	learn: 0.3819441	total: 50.4ms	remaining: 50.3s
500:	learn: 0.2094747	total: 25.4s	remaining: 25.3s
999:	learn: 0.1946784	total: 49.9s	remaining: 0us
CatBoost 모델링 완료


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  30 out of  30 | elapsed:  6.6min finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  30 out of  30 | elapsed:    0.4s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  30 out of  30 | elapsed:    2.1s finished


RandomForest_light 모델링 완료


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  30 out of  30 | elapsed:  3.1min finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  30 out of  30 | elapsed:    0.3s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.


ExtraTrees_light 모델링 완료


[Parallel(n_jobs=2)]: Done  30 out of  30 | elapsed:    1.8s finished


,Model,Time,val_RMSE,train_RMSE,Gap
0,LightGBM,3.418041,0.202442,0.219475,-0.017034
1,XGBoost,3.904450,0.200515,0.199243,0.001272
2,CatBoost,52.042789,0.195292,0.194678,0.000614
3,RandomForest_light,395.386572,0.199113,0.196129,0.002984
4,ExtraTrees_light,185.713025,0.207873,0.233739,-0.025866
